In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.ensemble import ExtraTreesRegressor, VotingRegressor
from catboost import CatBoostRegressor

In [2]:
possible_train_paths = [
    "../data/train_dataset.csv",
    "../data/raw/train_dataset.csv",
    "../data/processed/train_dataset.csv"
]

train_path = None

for path in possible_train_paths:
    if os.path.exists(path):
        train_path = path
        break

if train_path is None:
    raise FileNotFoundError(
        "Could not find train_dataset.csv. "
        "Check your data folder."
    )

print("Using:", train_path)

train_raw = pd.read_csv(train_path)

print("Shape:", train_raw.shape)
print(train_raw.columns.tolist())

Using: ../data/raw/train_dataset.csv
Shape: (150, 6)
['flow_rate_L_min', 'concentration_mol_L', 'inlet_temperature_K', 'length_m', 'jacket_temperature_K', 'overall_yield']


In [3]:
required_columns = [
    "flow_rate_L_min",
    "concentration_mol_L",
    "inlet_temperature_K",
    "length_m",
    "jacket_temperature_K",
    "overall_yield"
]

missing = [
    c for c in required_columns
    if c not in train_raw.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns found.")

All required columns found.


In [4]:
X_raw = train_raw.drop(columns=["overall_yield"])
y_raw = train_raw["overall_yield"]

print(X_raw.shape)
print(y_raw.shape)

(150, 5)
(150,)


In [5]:
def create_physics_features(df):

    data = df.copy()

    # -----------------------------------------
    # Thermal driving difference
    # -----------------------------------------
    data["temperature_difference_K"] = (
        data["inlet_temperature_K"]
        - data["jacket_temperature_K"]
    )

    # Magnitude of thermal difference
    data["temperature_difference_abs_K"] = (
        data["inlet_temperature_K"]
        - data["jacket_temperature_K"]
    ).abs()

    # -----------------------------------------
    # Relative thermal condition
    # -----------------------------------------
    data["temperature_ratio"] = (
        data["inlet_temperature_K"]
        / data["jacket_temperature_K"]
    )

    # -----------------------------------------
    # Flow / concentration relationship
    # -----------------------------------------
    data["flow_concentration_ratio"] = (
        data["flow_rate_L_min"]
        / data["concentration_mol_L"]
    )

    # -----------------------------------------
    # Reactor length relative to flow
    # -----------------------------------------
    data["length_flow_ratio"] = (
        data["length_m"]
        / data["flow_rate_L_min"]
    )

    # -----------------------------------------
    # Simple residence-time proxy
    # -----------------------------------------
    data["residence_time_proxy"] = (
        data["length_m"]
        / data["flow_rate_L_min"]
    )

    # -----------------------------------------
    # Thermal severity × residence proxy
    # -----------------------------------------
    data["thermal_residence_proxy"] = (
        data["temperature_difference_abs_K"]
        * data["residence_time_proxy"]
    )

    # -----------------------------------------
    # Flow × reactor length
    # -----------------------------------------
    data["flow_length_interaction"] = (
        data["flow_rate_L_min"]
        * data["length_m"]
    )

    # -----------------------------------------
    # Temperature × flow
    # -----------------------------------------
    data["temperature_flow_interaction"] = (
        data["inlet_temperature_K"]
        * data["flow_rate_L_min"]
    )

    # -----------------------------------------
    # Temperature × concentration
    # -----------------------------------------
    data["temperature_concentration_interaction"] = (
        data["inlet_temperature_K"]
        * data["concentration_mol_L"]
    )

    return data

In [6]:
X_physics = create_physics_features(X_raw)

print("Original features:", X_raw.shape[1])
print("Physics features:", X_physics.shape[1])

display(X_physics.head())

Original features: 5
Physics features: 15


,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,temperature_difference_K,temperature_difference_abs_K,temperature_ratio,flow_concentration_ratio,length_flow_ratio,residence_time_proxy,thermal_residence_proxy,flow_length_interaction,temperature_flow_interaction,temperature_concentration_interaction
0,33.09,3.68,357.75,19.87,383.79,-26.04,26.04,0.932150,8.991848,0.600484,0.600484,15.636591,657.4983,11837.9475,1316.5200
1,76.30,1.34,429.70,14.84,405.72,23.98,23.98,1.059105,56.940299,0.194495,0.194495,4.664000,1132.2920,32786.1100,575.7980
2,59.90,1.01,431.10,11.76,385.40,45.70,45.70,1.118578,59.306931,0.196327,0.196327,8.972154,704.4240,25822.8900,435.4110
3,49.90,2.21,445.61,22.85,367.74,77.87,77.87,1.211753,22.579186,0.457916,0.457916,35.657906,1140.2150,22235.9390,984.7981
4,16.70,3.95,458.91,4.56,374.13,84.78,84.78,1.226606,4.227848,0.273054,0.273054,23.149509,76.1520,7663.7970,1812.6945


In [7]:
cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

In [8]:
def create_winning_model():

    cat = CatBoostRegressor(
        iterations=600,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=9,
        subsample=1.0,
        verbose=0,
        random_seed=42
    )

    extra = ExtraTreesRegressor(
        n_estimators=700,
        max_depth=20,
        min_samples_leaf=2,
        min_samples_split=2,
        bootstrap=False,
        random_state=42
    )

    model = VotingRegressor(
        estimators=[
            ("cat", cat),
            ("extra", extra)
        ],
        weights=[3, 1]
    )

    return model

In [9]:
baseline_model = create_winning_model()

baseline_scores = -cross_val_score(
    baseline_model,
    X_raw,
    y_raw,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

baseline_rmse = baseline_scores.mean()

print("RAW FEATURE BASELINE")
print("--------------------")
print("RMSE:", round(baseline_rmse, 4))
print("STD :", round(baseline_scores.std(), 4))

RAW FEATURE BASELINE
--------------------
RMSE: 19.2808
STD : 2.495


In [10]:
physics_model = create_winning_model()

physics_scores = -cross_val_score(
    physics_model,
    X_physics,
    y_raw,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1
)

physics_rmse = physics_scores.mean()

print("PHYSICS-ENHANCED MODEL")
print("----------------------")
print("RMSE:", round(physics_rmse, 4))
print("STD :", round(physics_scores.std(), 4))

PHYSICS-ENHANCED MODEL
----------------------
RMSE: 19.9666
STD : 2.351


In [11]:
comparison = pd.DataFrame({
    "Model": [
        "Current architecture - raw features",
        "Current architecture - physics features"
    ],
    "RMSE": [
        baseline_rmse,
        physics_rmse
    ],
    "STD": [
        baseline_scores.std(),
        physics_scores.std()
    ]
})

comparison = comparison.sort_values("RMSE")

display(comparison)

,Model,RMSE,STD
0,Current architecture - raw features,19.280801,2.494957
1,Current architecture - physics features,19.966645,2.351008
